### ETL RAID -> AWS S3 - FASE 3

In [4]:
from pyspark.sql.functions import lit
from pyspark.sql import functions as f
from pyspark.sql.functions import *
from pyspark.sql import SparkSession
import pandas as pd
import logging

# FATURAMENTO EAD

In [5]:
faturamento_ead = spark.read.parquet('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/FATURAMENTO_EAD/')

In [6]:
## Exibir as colunas e os tipo de dados
faturamento_ead.printSchema() 

root
 |-- TIPO_BOLETO: string (nullable = true)
 |-- PERIODO: string (nullable = true)
 |-- CODIGO_PERIODO_MATRICULA: string (nullable = true)
 |-- CODIGO_UNIDADE: decimal(10,0) (nullable = true)
 |-- NOME_UNIDADE: string (nullable = true)
 |-- REGIONAL: string (nullable = true)
 |-- CODIGO_CURSO: string (nullable = true)
 |-- NOME_CURSO: string (nullable = true)
 |-- TIPO_CURSO: string (nullable = true)
 |-- TIPO_EAD: string (nullable = true)
 |-- EMAT_CD: string (nullable = true)
 |-- RA: string (nullable = true)
 |-- CODIGO_SITUACAO_ACADEMICA: string (nullable = true)
 |-- DESCRICAO_SITUACAO_ACADEMICA: string (nullable = true)
 |-- COD_SIT_ACAD_ATE_VCTO: string (nullable = true)
 |-- DESC_SIT_ACAD_ATE_VCTO: string (nullable = true)
 |-- FLAG_FILL_SIT_VENCIMENTO: decimal(1,0) (nullable = true)
 |-- DOCM_CD: decimal(18,8) (nullable = true)
 |-- CONTABILIZADO: string (nullable = true)
 |-- REF_MENS: string (nullable = true)
 |-- MES_COMP: string (nullable = true)
 |-- ANO_COMP: string 

In [10]:
faturamento_ead.select(max("DT_PAGAMENTO"),count("RA"),sum("VALOR_LIQUIDO_TITULO")).show(truncate=False)

+-------------------+---------+-------------------------+
|max(DT_PAGAMENTO)  |count(RA)|sum(VALOR_LIQUIDO_TITULO)|
+-------------------+---------+-------------------------+
|2022-05-13 14:43:55|1394156  |243903093.74             |
+-------------------+---------+-------------------------+



faturamento_ead = spark.read.parquet('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/FATURAMENTO_EAD/').limit(100)

In [13]:
faturamento_ead.columns

['TIPO_BOLETO',
 'PERIODO',
 'CODIGO_PERIODO_MATRICULA',
 'CODIGO_UNIDADE',
 'NOME_UNIDADE',
 'REGIONAL',
 'CODIGO_CURSO',
 'NOME_CURSO',
 'TIPO_CURSO',
 'TIPO_EAD',
 'EMAT_CD',
 'RA',
 'CODIGO_SITUACAO_ACADEMICA',
 'DESCRICAO_SITUACAO_ACADEMICA',
 'COD_SIT_ACAD_ATE_VCTO',
 'DESC_SIT_ACAD_ATE_VCTO',
 'FLAG_FILL_SIT_VENCIMENTO',
 'DOCM_CD',
 'CONTABILIZADO',
 'REF_MENS',
 'MES_COMP',
 'ANO_COMP',
 'MES_VENC',
 'ANO_VENC',
 'MES_PAG',
 'ANO_PAG',
 'DESCONTO_PRIM',
 'DESCONTO_SEC',
 'JUROS',
 'DESCONTO',
 'VALOR_LIQUIDO_TITULO',
 'VALOR_RECEBIDO',
 'MENS_VL_PROUNI',
 'DT_PAGAMENTO',
 'DT_VENCIMENTO',
 'TEMPO_PAGAMENTO',
 'TIPO_NOME_BAIXA']

In [14]:
#group by
#df_bi_carteira_cobranca =  (df_bi_carteira_cobranca.groupBy('COD_CONTRATO').agg(f.sum(f.col('VLR_LIQUIDO').cast('float')).alias('VLR_SALDO_DEV_VENC'), (f.min(f.to_date(f.col('DAT_VENCIMENTO'), 'dd/MM/yyyy'))).alias('DAT_MENOR_VENCIMENTO'))) 
df_faturamento_ead = (faturamento_ead.
                          agg(f.sum
                                  (f.col('VALOR_LIQUIDO_TITULO')
                                  ).alias('VLR_TOTAL')
                             )
                     )
    

In [15]:
df_faturamento_ead.show(truncate=False)

+------------+
|VLR_TOTAL   |
+------------+
|240435861.89|
+------------+



In [18]:
faturamento_ead.select(count("RA")).show(truncate=False)

+---------+
|count(RA)|
+---------+
|1229990  |
+---------+



In [17]:
faturamento_ead.groupBy("DESC_SIT_ACAD_ATE_VCTO").sum("VALOR_LIQUIDO_TITULO").show(truncate=False)

+---------------------------------------+-------------------------+
|DESC_SIT_ACAD_ATE_VCTO                 |sum(VALOR_LIQUIDO_TITULO)|
+---------------------------------------+-------------------------+
|Matricula Ativa                        |101501169.36             |
|Concluido                              |431515.68                |
|Cancelado por mudança de curso/turma   |4356.82                  |
|Matrícula Cancelada - Cadastro Indevido|525692.46                |
|Matrícula não efetuada e bloqueada     |3085695.55               |
|Transferência Externa                  |8597.71                  |
|Desistente                             |327304.80                |
|Vestibular                             |19016162.89              |
|Nao Efetuou Matric.                    |27109320.19              |
|Formando                               |14300098.43              |
|Matric. Canc. - Não formação de Turma  |33157.78                 |
|Concluído/Falecimento                  |491.08 

In [19]:
faturamento_ead.groupBy("DESC_SIT_ACAD_ATE_VCTO") \
    .agg(sum("VALOR_LIQUIDO_TITULO").alias("VALOR_LIQUIDO_TITULO"), \
         count("RA").alias("RA")
     ) \
    .show(truncate=False)

+---------------------------------------+--------------------+------+
|DESC_SIT_ACAD_ATE_VCTO                 |VALOR_LIQUIDO_TITULO|RA    |
+---------------------------------------+--------------------+------+
|Matricula Ativa                        |101501169.36        |363700|
|Concluido                              |431515.68           |3421  |
|Cancelado por mudança de curso/turma   |4356.82             |81    |
|Matrícula Cancelada - Cadastro Indevido|525692.46           |3625  |
|Matrícula não efetuada e bloqueada     |3085695.55          |13523 |
|Transferência Externa                  |8597.71             |53    |
|Desistente                             |327304.80           |1963  |
|Vestibular                             |19016162.89         |289344|
|Nao Efetuou Matric.                    |27109320.19         |104281|
|Formando                               |14300098.43         |65924 |
|Matric. Canc. - Não formação de Turma  |33157.78            |304   |
|Concluído/Falecimen

### Acordos Olimpo

In [20]:
acordos_olimpo = spark.read.parquet('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/ACORDOS_OLIMPO/')

In [21]:
acordos_olimpo.count()

209

In [23]:
acordos_olimpo.select(sum("VALOR_ENTRADA")).show(truncate=False)

+------------------+
|sum(VALOR_ENTRADA)|
+------------------+
|434123.29         |
+------------------+



In [27]:
acordos_olimpo.select(max("DT_PAGAMENTO")).show(truncate=False)

+-------------------+
|max(DT_PAGAMENTO)  |
+-------------------+
|2022-05-03 18:58:56|
+-------------------+



In [26]:
acordos_olimpo.groupBy("DT_PAGAMENTO") \
    .agg(sum("VALOR_ENTRADA").alias("VALOR_ENTRADA"), \
         count("RA").alias("RA")
     ) \
    .show(truncate=False)

+-------------------+-------------+---+
|DT_PAGAMENTO       |VALOR_ENTRADA|RA |
+-------------------+-------------+---+
|2022-05-03 18:50:03|310.08       |1  |
|2022-05-02 17:49:35|2781.89      |1  |
|2022-05-02 21:06:17|342.12       |1  |
|2022-05-02 11:36:44|651.70       |1  |
|2022-04-26 14:25:29|4413.19      |1  |
|2022-05-02 15:04:42|69.86        |1  |
|2022-05-02 20:24:51|68.20        |1  |
|2022-05-02 13:27:56|307.56       |1  |
|2022-05-03 13:03:01|35.89        |1  |
|2022-05-02 15:08:04|719.99       |1  |
|2022-05-02 12:53:41|722.17       |1  |
|2022-05-03 16:31:21|423.62       |1  |
|2022-05-03 13:12:01|774.65       |1  |
|2022-04-29 22:47:34|858.84       |1  |
|2022-05-02 20:33:45|572.67       |1  |
|2022-05-02 17:27:55|1076.90      |1  |
|2022-05-03 15:18:44|1106.69      |1  |
|2022-05-03 15:05:12|46.94        |1  |
|2022-05-02 16:01:28|28.21        |1  |
|2022-05-02 19:25:20|570.61       |1  |
+-------------------+-------------+---+
only showing top 20 rows



### Faturamento Presencial

In [7]:
faturamento_pres = spark.read.parquet('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/FATURAMENTO_PRESENCIAL/')

In [8]:
faturamento_pres.printSchema()

root
 |-- BASE: string (nullable = true)
 |-- TIPO_BOLETO: string (nullable = true)
 |-- RA: decimal(9,0) (nullable = true)
 |-- CODIGO_CURSO: decimal(5,0) (nullable = true)
 |-- CODIGO_UNIDADE: decimal(6,0) (nullable = true)
 |-- NOME_UNIDADE: string (nullable = true)
 |-- MARCA: string (nullable = true)
 |-- REGIONAL: string (nullable = true)
 |-- NOME_CURSO: string (nullable = true)
 |-- TIPO_CURSO: string (nullable = true)
 |-- PERIODO: string (nullable = true)
 |-- CODIGO_PERIODO_MATRICULA: decimal(5,0) (nullable = true)
 |-- COD_SIT_ACAD_ATE_VCTO: string (nullable = true)
 |-- DESC_SIT_ACAD_ATE_VCTO: string (nullable = true)
 |-- MES_COMP: decimal(2,0) (nullable = true)
 |-- ANO_COMP: decimal(4,0) (nullable = true)
 |-- DT_PAGAMENTO_OLD: timestamp (nullable = true)
 |-- FLAG_DT_PAGAMENTO_OLD: string (nullable = true)
 |-- DT_PAGAMENTO: timestamp (nullable = true)
 |-- FLAG_DT_PAGAMENTO: string (nullable = true)
 |-- DT_BAIXA: timestamp (nullable = true)
 |-- TRARETDATAPROC: times

In [22]:
faturamento_pres.select(max("DT_PAGAMENTO"),count("RA"),sum("VALOR_LIQUIDO_TITULO")).show(truncate=False)

+-------------------+---------+-------------------------+
|max(DT_PAGAMENTO)  |count(RA)|sum(VALOR_LIQUIDO_TITULO)|
+-------------------+---------+-------------------------+
|2022-05-19 03:00:00|842348   |502913463.56             |
+-------------------+---------+-------------------------+



### Recebimento EAD

In [16]:
recebimento_ead = spark.read.parquet('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/RECEBIMENTO_EAD/')

In [17]:
recebimento_ead.printSchema()

root
 |-- TIPO_BOLETO: string (nullable = true)
 |-- PERIODO: string (nullable = true)
 |-- CODIGO_PERIODO_MATRICULA: string (nullable = true)
 |-- CODIGO_UNIDADE: decimal(10,0) (nullable = true)
 |-- NOME_UNIDADE: string (nullable = true)
 |-- REGIONAL: string (nullable = true)
 |-- CODIGO_CURSO: string (nullable = true)
 |-- NOME_CURSO: string (nullable = true)
 |-- TIPO_CURSO: string (nullable = true)
 |-- TIPO_EAD: string (nullable = true)
 |-- EMAT_CD: string (nullable = true)
 |-- RA: string (nullable = true)
 |-- CODIGO_SITUACAO_ACADEMICA: string (nullable = true)
 |-- DESCRICAO_SITUACAO_ACADEMICA: string (nullable = true)
 |-- COD_SIT_ACAD_ATE_VCTO: string (nullable = true)
 |-- DESC_SIT_ACAD_ATE_VCTO: string (nullable = true)
 |-- FLAG_FILL_SIT_VENCIMENTO: decimal(1,0) (nullable = true)
 |-- DOCM_CD: decimal(18,0) (nullable = true)
 |-- CONTABILIZADO: string (nullable = true)
 |-- REF_MENS: string (nullable = true)
 |-- MES_COMP: string (nullable = true)
 |-- ANO_COMP: string 

In [10]:
recebimento_ead.select(max("DT_PAGAMENTO"),count("DOCM_CD"),sum("VALOR_RECEBIDO")).show(truncate=False)

+-------------------+--------------+-------------------+
|max(DT_PAGAMENTO)  |count(DOCM_CD)|sum(VALOR_RECEBIDO)|
+-------------------+--------------+-------------------+
|2022-05-10 18:12:39|472010        |97280654.51        |
+-------------------+--------------+-------------------+



### Recebimento Presencial

In [18]:
recebimento_pres = spark.read.parquet('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/RECEBIMENTO_PRESENCIAL/')

In [19]:
recebimento_pres.printSchema()

root
 |-- BASE: string (nullable = true)
 |-- TIPO_BOLETO: string (nullable = true)
 |-- RA: decimal(9,0) (nullable = true)
 |-- CODIGO_CURSO: decimal(5,0) (nullable = true)
 |-- CODIGO_UNIDADE: decimal(6,0) (nullable = true)
 |-- NOME_UNIDADE: string (nullable = true)
 |-- MARCA: string (nullable = true)
 |-- REGIONAL: string (nullable = true)
 |-- NOME_CURSO: string (nullable = true)
 |-- TIPO_CURSO: string (nullable = true)
 |-- PERIODO: string (nullable = true)
 |-- CODIGO_PERIODO_MATRICULA: decimal(5,0) (nullable = true)
 |-- COD_SIT_ACAD_ATE_VCTO: string (nullable = true)
 |-- DESC_SIT_ACAD_ATE_VCTO: string (nullable = true)
 |-- MES_COMP: decimal(2,0) (nullable = true)
 |-- ANO_COMP: decimal(4,0) (nullable = true)
 |-- DT_PAGAMENTO_OLD: timestamp (nullable = true)
 |-- FLAG_DT_PAGAMENTO_OLD: string (nullable = true)
 |-- DT_PAGAMENTO: timestamp (nullable = true)
 |-- FLAG_DT_PAGAMENTO: string (nullable = true)
 |-- DT_BAIXA: timestamp (nullable = true)
 |-- TRARETDATAPROC: times

In [11]:
recebimento_pres.select(max("DT_PAGAMENTO"),count("DOCNUM"),sum("VALOR_RECEBIDO")).show(truncate=False)

+-------------------+-------------+-------------------+
|max(DT_PAGAMENTO)  |count(DOCNUM)|sum(VALOR_RECEBIDO)|
+-------------------+-------------+-------------------+
|2022-05-19 03:00:00|536391       |82584263.24        |
+-------------------+-------------+-------------------+



In [156]:
bi_isencao_credito_maior_debito = spark.read.parquet('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/BI_ISENCAO_CREDITO_MAIOR_DEBITO/')

In [157]:
bi_carteira_cobranca = spark.read.parquet('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/BI_CARTEIRA_COBRANCA/')

In [158]:
bi_isencao = spark.read.parquet('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/ISENCAO_RAID/')

In [159]:
def join_isencao(bi_carteira_cobranca, isencao_raid):
    df_isencao = (bi_carteira_cobranca.join(isencao_raid,
                                  bi_carteira_cobranca['COD_CONTRATO'] == isencao_raid['COD_CONTRATO_ISENCAO'], "inner"))    
    df_isencao = df_isencao.select(f.col('COD_CONTRATO_ISENCAO'),f.col('DSC_MOTIVO_ISENCAO'))
    
    return df_isencao

In [160]:
def join_isencao_matricula_nao_paga(bi_carteira_cobranca, bi_isencao_matricula_nao_paga):
    df_isencao_matricula = (bi_carteira_cobranca.join(bi_isencao_matricula_nao_paga,
                                  bi_carteira_cobranca['COD_CPFCGC_PES'] == bi_isencao_matricula_nao_paga['COD_RA_IMNP'], "inner"))
    df_isencao_matricula = df_isencao_matricula.select(f.col('COD_CONTRATO').alias('COD_CONTRATO_MAT'),f.col('DSC_MOTIVO_MNP').alias('DSC_MOTIVO_MAT'))
    
    return df_isencao_matricula

In [161]:
def join_isencao_cmd(bi_carteira_cobranca, bi_isencao_credito_maior_debito):
    df_isencao_cmd = (bi_carteira_cobranca.join(bi_isencao_credito_maior_debito,
                                  bi_carteira_cobranca['COD_CPFCGC_PES'] == bi_isencao_credito_maior_debito['COD_RA_ICMD'], "inner"))
    df_isencao_cmd = df_isencao_cmd.select(f.col('COD_CONTRATO').alias('COD_CONTRATO_CRE'),f.col('STA_COBRANCA_CMD').alias('DSC_MOTIVO_CRE'))
    
    return df_isencao_cmd

In [162]:
df_isencao_cmd = join_isencao_cmd(bi_carteira_cobranca, bi_isencao_credito_maior_debito)
df_isencao_matricula = join_isencao_matricula_nao_paga(bi_carteira_cobranca, bi_isencao_matricula_nao_paga)
df_isencao = join_isencao(bi_carteira_cobranca, bi_isencao)
df_cd_carteira = bi_carteira_cobranca

In [163]:
df_isencao_matricula.printSchema()

root
 |-- COD_CONTRATO_MAT: string (nullable = true)
 |-- DSC_MOTIVO_MAT: string (nullable = true)



In [164]:
df_isencao.printSchema()

root
 |-- COD_CONTRATO_ISENCAO: string (nullable = true)
 |-- DSC_MOTIVO_ISENCAO: string (nullable = true)



In [165]:
df_cd_carteira = (df_cd_carteira.join(df_isencao,
                                  df_cd_carteira['COD_CONTRATO'] == df_isencao['COD_CONTRATO_ISENCAO'], "left"))

In [166]:
df_cd_carteira = (df_cd_carteira.join(df_isencao_matricula,
                                  df_cd_carteira['COD_CONTRATO'] == df_isencao_matricula['COD_CONTRATO_MAT'], "left"))

In [167]:
    df_cd_carteira = (df_cd_carteira.join(df_isencao_cmd,
                                      df_cd_carteira['COD_CONTRATO'] == df_isencao_cmd['COD_CONTRATO_CRE'], "left"))

In [171]:
df_cd_carteira = (
    df_cd_carteira
    .withColumn("DSC_SITUACAO_COBRANCA", 
        f.when(f.col("DSC_MOTIVO_MAT").isNotNull(), 'MATRICULA NAO PAGA')
        .when((f.col("DSC_MOTIVO_ISENCAO").isNotNull()),f.col("DSC_MOTIVO_ISENCAO"))
        .when((f.col("CLS_SITUACAO_CONTRATO_ALUNO").isin('12','14','17','20','22','23','33','36','37','Err')),"FORA COBRANCA SITUACAO ACADEMICA")
        .when((f.col("DSC_MOTIVO_CRE").isNotNull()), 'CREDITO MAIOR QUE DEBITO')
        .otherwise(f.lit('EM COBRANCA'))
))

In [172]:
df_cd_carteira.select(f.col('DSC_SITUACAO_COBRANCA')).distinct().show()

+---------------------+
|DSC_SITUACAO_COBRANCA|
+---------------------+
| TRAVA SISTEMICA C...|
| TRAVA SISTEMICA C...|
| TRAVA SISTEMICA C...|
| TRAVA SISTEMICA C...|
| TRAVA SISTEMICA C...|
| TRAVA SISTEMICA C...|
| TRAVA SISTEMICA C...|
| TRAVA SISTEMICA C...|
| TRAVA SISTEMICA C...|
| TRAVA SISTEMICA C...|
| TRAVA SISTEMICA C...|
| TRAVA SISTEMICA C...|
| TRAVA SISTEMICA C...|
| TRAVA SISTEMICA C...|
| TRAVA SISTEMICA C...|
| TRAVA SISTEMICA C...|
| TRAVA SISTEMICA C...|
| TRAVA SISTEMICA C...|
| TRAVA SISTEMICA C...|
| TRAVA SISTEMICA C...|
+---------------------+
only showing top 20 rows



In [147]:
credito = (df_cd_carteira.filter(f.col('DSC_SITUACAO_COBRANCA') == 'CREDITO MAIOR QUE DEBITO'))

In [149]:
matricula = (df_cd_carteira.filter(f.col('DSC_SITUACAO_COBRANCA') == "MATRICULA NAO PAGA"))

In [173]:
isencao = (df_cd_carteira.filter(f.col('DSC_SITUACAO_COBRANCA') == "FORA COBRANCA SITUACAO ACADEMICA"))

In [174]:
isencao.count()

480619

In [151]:
df_cd_carteira.select(f.col('DSC_SITUACAO_COBRANCA')).distinct().show()

+---------------------+
|DSC_SITUACAO_COBRANCA|
+---------------------+
|          EM COBRANCA|
| FORA DE COBRANCA ...|
+---------------------+



In [153]:
df_isencao.count()

143631

In [142]:
df_cd_carteira.select(f.col('DSC_SITUACAO_COBRANCA')).show()

+---------------------+
|DSC_SITUACAO_COBRANCA|
+---------------------+
|                 null|
|                 null|
|                 null|
|                 null|
|                 null|
|                 null|
|                 null|
|                 null|
|                 null|
|                 null|
|                 null|
|                 null|
|                 null|
|                 null|
|                 null|
| FORA DE COBRANCA ...|
|                 null|
|                 null|
|                 null|
| FORA DE COBRANCA ...|
+---------------------+
only showing top 20 rows



In [ ]:
df_cd_carteira = df_cd_carteira.select(f.col("*"), f.expr("CASE WHEN COD_STCO_ACDM in ('C','P') THEN 'ATIVO' " + 
                                                          "WHEN COD_STCO_ACDM in ('D') THEN 'INATIVO'" +
                                                          "ELSE 'NAO ENCONTRADO' END").alias("STA_STCO_ACDM"))

In [ ]:
#nova coluna - ISENCAO
df_cd_carteira =  (df_cd_carteira.withColumn("DSC_SITUACAO_COBRANCA", f.when(f.col("DSC_SITUACAO_COBRANCA").isNull(),
                                                     f.lit("EM COBRANCA"))
                                            .otherwise(
                                                f.col('DSC_SITUACAO_COBRANCA')
                                            )))

In [83]:
df_cd_carteira.printSchema()

root
 |-- NOM_AGENCIA_EXTERNA: string (nullable = true)
 |-- DAT_PROCESSAMENTO: date (nullable = true)
 |-- CLS_MARCA_CONTRATO: string (nullable = true)
 |-- COD_FILIAL: string (nullable = true)
 |-- DSC_FILIAL: string (nullable = true)
 |-- CLS_SITUACAO_CONTRATO_ALUNO: string (nullable = true)
 |-- DSC_SITUACAO_CONTRATO: string (nullable = true)
 |-- COD_CLIENTE: string (nullable = true)
 |-- COD_CONTRATO: string (nullable = true)
 |-- COD_CONTRATO_ORIGEM: string (nullable = true)
 |-- COD_CPFCGC_PES: string (nullable = true)
 |-- NOM_CLIENTE: string (nullable = true)
 |-- CLS_SCORE_COLLECTION: string (nullable = true)
 |-- CLS_CLASSE_DEBITO: string (nullable = true)
 |-- CLS_CLUSTER: string (nullable = true)
 |-- CLS_SCORE_GLOBAL: string (nullable = true)
 |-- CLS_SCORE_ENGAJAMENTO: string (nullable = true)
 |-- DAT_INICIO_CURSO: string (nullable = true)
 |-- DAT_FIM_CURSO: string (nullable = true)
 |-- DAT_INICIO_MATRICULA: string (nullable = true)
 |-- DAT_FIM_MATRICULA: string (nu

In [81]:
df_cd_carteira = (df_cd_carteira.withColumn("DSC_STA_CBRO", f.when(f.col("DSC_MOTIVO").isNull(),"EM COBRANCA")
                                            .otherwise(
                                                f.col("DSC_MOTIVO")
                                            )))                            

In [51]:
df_isencao_total.printSchema()

root
 |-- COD_CONTRATO_ISENCAO: string (nullable = true)
 |-- DSC_MOTIVO_ISENCAO: string (nullable = true)



In [48]:
df_cd_carteira.select(f.col('DSC_STA_CBRO')).show()

+----------------+
|    DSC_STA_CBRO|
+----------------+
|     EM COBRANCA|
|     EM COBRANCA|
|     EM COBRANCA|
|FORA DE COBRANCA|
|     EM COBRANCA|
|     EM COBRANCA|
|     EM COBRANCA|
|     EM COBRANCA|
|     EM COBRANCA|
|     EM COBRANCA|
|     EM COBRANCA|
|     EM COBRANCA|
|     EM COBRANCA|
|     EM COBRANCA|
|     EM COBRANCA|
|     EM COBRANCA|
|     EM COBRANCA|
|     EM COBRANCA|
|     EM COBRANCA|
|     EM COBRANCA|
+----------------+
only showing top 20 rows



In [ ]:
  
    df_cd_carteira = (df_carteira_cobranca.join(df_cd_zeus,
                                  df_carteira_cobranca['COD_CPFCGC_PES'] == df_cd_zeus['COD_RA_ZEUS'], "left"))
    
    

    
    
    #nova coluna - ISENCAO
    df_cd_carteira =  (df_cd_carteira.withColumn("DSC_SITUACAO_COBRANCA", f.when(f.col("CLS_SITUACAO_CONTRATO_ALUNO").isin('12','14','17','20','22','23','33','36','37','Err'),
                                                     f.lit("FORA DE COBRANCA - SITUACAO ACADEMICA"))
                                            .otherwise(
                                                f.col('DSC_MOTIVO_ISENCAO')
                                            ))) 
    
    
    #nova coluna - ISENCAO
    df_cd_carteira =  (df_cd_carteira.withColumn("DSC_SITUACAO_COBRANCA", f.when(f.col("DSC_SITUACAO_COBRANCA").isNull(),
                                                     f.lit("EM COBRANCA"))
                                            .otherwise(
                                                f.col('DSC_SITUACAO_COBRANCA')
                                            )))
                                            
    
                                            
    #NEGATIVADO SERASA 
    df_cd_carteira =  (df_cd_carteira.withColumn("DSC_NEGATIVADO_SERASA", f.when(f.col("NOM_ACAO") == f.lit("Negativacao Serasa"),
                                                     f.lit("SIM"))
                                            .otherwise(
                                                f.lit("NAO")
                                            )))
                                            
    #NEGATIVADO BOA VISTA
    df_cd_carteira =  (df_cd_carteira.withColumn("DSC_NEGATIVADO_BOA_VISTA", f.when(f.col("NOM_ACAO") == f.lit("Negativacao Boa Vista"),
                                                     f.lit("SIM"))
                                            .otherwise(
                                                f.lit("NAO")
                                            )))
                                            
                                            
    df_cd_carteira = df_cd_carteira.dropDuplicates(['COD_CONTRATO'])
    
    df_cd_carteira = df_cd_carteira.dropDuplicates(['COD_CPFCGC_PES'])
    
    return df_cd_carteira